# Importing packages

In [1]:
# Python packages.
import csv
import pathlib
import typing

# External packages.
import numpy
from plotly import (
    colors,
    graph_objects,
    subplots,
)
from tqdm import notebook

# Scoring function

## Absolute

In [2]:
def absolute(to_match: int, pharmacophore_generated_molecule: int) -> typing.Callable[[float], float]:
    """Generated a function for given parameters.

    Parameters
    ----------
    to_match : `int`
        The nimber of pharmacophore we are trying to match to.

    pharmacophore_generated_molecule : `int`
        The total number of pharmacophore of the generated drug.

    Return
    ------
    `typing.Callable[[float], float]`
        The scoring function, taking in arguments "the number of matching pharmacophore"
        and returning a score value for the fixed previous parameters.
    """
    return lambda match : 1 / (
        1 + abs (
            (to_match - match)
            + max(0, pharmacophore_generated_molecule - to_match)
        )
    )

## Gaussian

In [3]:
def gaussian(to_match: int, pharmacophore_generated_molecule: int, sd: float) -> typing.Callable[[float], float]:
    """Generated a function for given parameters.

    Parameters
    ----------
    to_match : `int`
        The nimber of pharmacophore we are trying to match to.

    pharmacophore_generated_molecule : `int`
        The total number of pharmacophore of the generated drug.

    sd : `float`
        The standard deviation

    Return
    ------
    `typing.Callable[[float], float]`
        The scoring function, taking in arguments "the number of matching pharmacophore"
        and returning a score value for the fixed previous parameters.
    """
    return lambda match : numpy.exp(
        -(to_match - match + max(0, pharmacophore_generated_molecule - to_match)) ** 2
        / (2 * sd ** 2)
    )

## 3D plotting

In [4]:
to_match: int = 6
match: numpy.ndarray = numpy.arange(0, to_match, 0.05)
over_generated: numpy.ndarray = numpy.arange(0, 20, 0.05)

absolute_score: list = []
gaussian_score: list = []

font_family: str = "Roboto"
font_size: float = 30

for match_val in notebook.tqdm(match, desc="Computing score"):
    sub_absolute_score: list = []
    sub_gaussian_score: list = []

    for over_val in over_generated:
        if over_val < match_val:
            sub_absolute_score.append(-1)
            sub_gaussian_score.append(-1)
            continue

        f_x = absolute(to_match, over_val)
        sub_absolute_score.append(f_x(match_val))

        f_x = gaussian(to_match, over_val, sd=5)
        sub_gaussian_score.append(f_x(match_val))

    absolute_score.append(sub_absolute_score)
    gaussian_score.append(sub_gaussian_score)

absolute_score: numpy.ndarray = numpy.array(absolute_score)
gaussian_score: numpy.ndarray = numpy.array(gaussian_score)

figure: graph_objects.Figure = subplots.make_subplots(
    rows=1,
    cols=2,
    specs=[[{"is_3d": True}] * 2],
)

line_setup: dict[str, str | float] = {
    "color": "#777777",
    "width": 0.5,
}
step: int = 20
ratio: float = over_generated.shape[0] / match.shape[0]

for column, z in enumerate([absolute_score, gaussian_score], 1):
    figure.add_trace(
        graph_objects.Surface(
            x = over_generated,
            y = match,
            z = z,
            colorscale = "magma",
            cmin = 0,
            cmax = 1,
        
            colorbar = {
                "tickfont": {
                    "family": font_family + " Light",
                    "size": font_size * 0.7,
                },

                "orientation": "v",
                "outlinecolor": "black",
                "outlinewidth": 1,
                "ticks": "outside",
        
                # Width and height.
                "thickness": 12,
                "len": 0.3,
        
                # Position.
                "x": 0.95,
                "y": 0.90,
                "xanchor": "left",
                "yanchor": "top",
            }
        ),
        row=1,
        col=column,
    )

    index_list: list[int] = list(range(0, over_generated.shape[0], step)) + [-1]
    
    for index in index_list:
        figure.add_trace(
            graph_objects.Scatter3d(
                x = [over_generated[index]] * len(index_list),
                y = list(match[::step]) + [match[-1]],
                z = list(z[::step, index]) + [z[-1, index]],
                showlegend = False,
                mode = "lines",
                line = line_setup,
                hoverinfo="skip",
            ),
            row=1,
            col=column,
        )
    
    index_list = list(range(0, match.shape[0], int(step / ratio))) + [-1]
    
    for index in index_list:
        figure.add_trace(
            graph_objects.Scatter3d(
                x = list(over_generated[::step]) + [over_generated[-1]],
                y = [match[index]] * len(index_list),
                z = list(z[index, ::step]) + [z[index, -1]],
                showlegend = False,
                mode = "lines",
                line = line_setup,
                hoverinfo="skip",
            ),
            row=1,
            col=column,
        )

axis_setup: dict = {
    "linewidth": 4,
    "ticks": "outside",
    "tickfont": {
        "family": font_family + " Light",
        "size": font_size / 2,
    },
    "mirror": True,
    "showgrid": True,
    "gridcolor": "black",
    "gridwidth": 1,
}

scene_setup: dict = {
    "xaxis": axis_setup |{
        "range": [0, 20],
        "title": {
            "text": "<b>Generated</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },

    "yaxis": axis_setup |{
        "range": [0, to_match],
        "title": {
            "text": "<b>Corresponding</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },

    "zaxis": axis_setup |{
        "range": [0, 1],
        "title": {
            "text": "<b>Score</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },

    "camera": {
        "eye": {"x": 1.7, "y": 1.7, "z": 1.7},
        "projection": {"type": "perspective"},
    }
}

annotation_setup: dict = {
    "font": {
        "family": font_family,
        "size": font_size,
    },
    "showarrow": False,
    "xref": "paper",
    "yref": "paper",
    "xanchor": "left",
    "yanchor": "top"
}

figure.update_layout({
    "template": "simple_white",
    "width": 1_500,
    "height": 700,
    "margin": {"l": 0, "r": 0, "t": 0, "b": 0},
    "scene": scene_setup,
    "scene2": scene_setup,
    "annotations": [
        annotation_setup |{"text": "<b>Inverse function</b>", "x": 0.0, "y": 0.90},
        annotation_setup |{"text": "<b>Gaussian function</b>", "x": 0.5, "y": 0.90},
    ],
})

figure.write_html(
    full_html=False,
    include_plotlyjs="/hugo_presentation/plotly.js",
    file="scoring_function.html",
    default_height="100%",
    default_width="100%",
)

widget = graph_objects.FigureWidget(figure)
widget

Computing score:   0%|          | 0/120 [00:00<?, ?it/s]

FigureWidget({
    'data': [{'cmax': 1,
              'cmin': 0,
              'colorbar': {'len': 0.3,
                           'orientation': 'v',
                           'outlinecolor': 'black',
                           'outlinewidth': 1,
                           'thickness': 12,
                           'tickfont': {'family': 'Roboto Light', 'size': 21.0},
                           'ticks': 'outside',
                           'x': 0.95,
                           'xanchor': 'left',
                           'y': 0.9,
                           'yanchor': 'top'},
              'colorscale': [[0.0, '#000004'], [0.1111111111111111, '#180f3d'],
                             [0.2222222222222222, '#440f76'], [0.3333333333333333,
                             '#721f81'], [0.4444444444444444, '#9e2f7f'],
                             [0.5555555555555556, '#cd4071'], [0.6666666666666666,
                             '#f1605d'], [0.7777777777777778, '#fd9668'],
                  

In [5]:
widget.layout.scene.camera

layout.scene.Camera({
    'eye': {'x': 1.7, 'y': 1.7, 'z': 1.7}, 'projection': {'type': 'perspective'}
})

# Docking score

In [6]:
font_family: str = "Roboto"
font_size: float = 30

score: list[float] = []

with open("docking_score.csv", mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        value: str | None = row.get("AFFINITY_KCAL_PER_MOL")

        if not (value and value.strip()):
            continue

        if float(value) >= 100:
            continue

        score.append(float(value))

figure: graph_objects.Figure = graph_objects.Figure()

figure.add_trace(
    graph_objects.Violin(
        x=score,
        orientation="h",
        box_visible=True,
        meanline_visible=True,
        fillcolor="white",
        line_color="black",
        line_width=1,
        marker={"color": "black"},
        showlegend=False,
    )
)

# Horizontal red line for the celecoxib.
celecoxib_score: float = -4.319

figure.add_vline(
    x=celecoxib_score,
    line={
        "color": "red",
        "width": 2,
    },
    opacity=1,
    annotation_text=f"celecoxib ({celecoxib_score})",
    annotation_position="top left",
    annotation_font={
        "family": font_family + " Light",
        "size": font_size * 2 / 3,
        "color": "red",
    },
)

axis_setup: dict = {
    "linewidth": 4,
    "ticks": "outside",
    "tickfont": {
        "family": font_family + " Light",
        "size": font_size / 2,
    },
    "mirror": True,
    "showgrid": False,
}

figure.update_layout({
    "template": "simple_white",
    "width": 1_500,
    "height": 700,
    "xaxis": axis_setup | {
        "title": {
            "text": "<b>Docking score (kcal · mol<sup>-1</sup>)</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },
    "yaxis": axis_setup | {
        "showticklabels": False,
        "ticks": "",
        "title": {
            "text": "",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },
})

figure.write_html(
    full_html=False,
    include_plotlyjs="/hugo_presentation/plotly.js",
    file="docking_score.html",
    default_height="100%",
    default_width="100%",
)

widget = graph_objects.FigureWidget(figure)
widget

FigureWidget({
    'data': [{'box': {'visible': True},
              'fillcolor': 'white',
              'line': {'color': 'black', 'width': 1},
              'marker': {'color': 'black'},
              'meanline': {'visible': True},
              'orientation': 'h',
              'showlegend': False,
              'type': 'violin',
              'uid': '527f7c9e-7688-46c9-9e86-85c780924a51',
              'x': [-148.368, -125.353, -52.773, ..., 67.627, 74.907, 85.503]}],
    'layout': {'annotations': [{'font': {'color': 'red', 'family': 'Roboto Light', 'size': 20.0},
                                'showarrow': False,
                                'text': 'celecoxib (-4.319)',
                                'x': -4.319,
                                'xanchor': 'right',
                                'xref': 'x',
                                'y': 1,
                                'yanchor': 'top',
                                'yref': 'y domain'}],
               'height': 

# Synthemol score

In [7]:
font_family: str = "Roboto"
font_size: float = 30

score: list[float] = []
roll_out: list[int] = []

with open("synthemol_result.csv", mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        roll_out_row: str | None = row.get("rollout_num")
        score_row: str | None = row.get("pharmacophore")

        if not (roll_out_row and roll_out_row.strip()):
            continue

        if not (score_row and score_row.strip()):
            continue

        roll_out.append(int(roll_out_row))
        score.append(float(score_row))

figure: graph_objects.Figure = subplots.make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    column_widths=[2/3, 1/3],
    horizontal_spacing=0,
)

figure.add_trace(
    graph_objects.Scatter(
        x=roll_out,
        y=score,
        mode="markers",
        marker={
            "color": "black",
            "size": 6,
        },
        showlegend=False,
    ),
    row=1,
    col=1,
)

figure.add_trace(
    graph_objects.Violin(
        y=score,
        orientation="v",
        box_visible=True,
        meanline_visible=True,
        fillcolor="white",
        line_color="black",
        line_width=1,
        marker={"color": "black"},
        showlegend=False,
    ),
    row=1,
    col=2,
)

axis_setup: dict = {
    "linewidth": 4,
    "ticks": "outside",
    "tickfont": {
        "family": font_family + " Light",
        "size": font_size / 2,
    },
    "mirror": True,
    "showgrid": False,
}

figure.update_layout({
    "template": "simple_white",
    "width": 1_500,
    "height": 700,
    "hovermode": "y unified",
    "xaxis": axis_setup | {
        "title": {
            "text": "<b>Roll-out</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },
    "xaxis2": axis_setup | {
        "showticklabels": False,
        "ticks": "",
        "title": {
            "text": "",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },
    "yaxis": axis_setup | {
        "mirror": False,
        "title": {
            "text": "<b>Docking score (kcal · mol<sup>-1</sup>)</b>",
            "font": {"size": font_size, "family": font_family + " Black"},
        }
    },
    "yaxis2": axis_setup | {
        "ticks": "",
        "title": {"text": ""},
    },
})

figure.write_html(
    full_html=False,
    include_plotlyjs="/hugo_presentation/plotly.js",
    file="synthemol_result.html",
    default_height="100%",
    default_width="100%",
)

widget = graph_objects.FigureWidget(figure)
widget

FigureWidget({
    'data': [{'marker': {'color': 'black', 'size': 6},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter',
              'uid': 'e041f437-822a-4390-847d-00c90a79166a',
              'x': [13460, 13942, 14463, ..., 13974, 13260, 17428],
              'xaxis': 'x',
              'y': [0.37531109885139957, 0.37531109885139957, 0.37531109885139957,
                    ..., 5.5346100717010135e-12, 1.2853372251336503e-12,
                    1.2853372251336503e-12],
              'yaxis': 'y'},
             {'box': {'visible': True},
              'fillcolor': 'white',
              'line': {'color': 'black', 'width': 1},
              'marker': {'color': 'black'},
              'meanline': {'visible': True},
              'orientation': 'v',
              'showlegend': False,
              'type': 'violin',
              'uid': '96bf2f2b-92a1-4468-a3cf-f84d0b8dc9df',
              'xaxis': 'x2',
              'y': [0.3753110988